# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to explore the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. You will learn to load, inspect, and analyze structured datasets defined by [Croissant schemas](https://mlcommons.org/croissant-schema/).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print out high-level metadata information
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
List available record sets, their `@id`s, and fields within each record set.

**Tip:** Each record set and field should be referenced by its `@id`, which is unique and stable in the Croissant schema.


In [ ]:
# List all record sets and their fields with field @id references

rs_list = list(metadata.record_sets)
print(f"Found {len(rs_list)} record sets in the dataset.\n")
for rs in rs_list:
    print(f"Record Set: {rs.name} (ID: {rs.id})")
    # Print each field: name and id
    if hasattr(rs, 'fields'):
        for f in rs.fields:
            print(f"  Field: {f.name} (ID: {f.id}) | Type: {getattr(f, 'data_type', 'N/A')}")
    print("")

## 3. Data Extraction
Load data from one or more record sets into DataFrames.

**Instructions:**
- Use the `@id` of record sets you want to extract data from.
- The keys and columns in the returned records are also their field `@id`s. For clarity, field names are shown above in the overview.


In [ ]:
# --- Prepare variable lists of record set IDs (referenced by @id only) ---
# Extract all available record sets
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for rsid in record_set_ids:
    # Load all records for that record set
    df = pd.DataFrame(list(dataset.records(record_set=rsid)))
    dataframes[rsid] = df
    print(f"Loaded {len(df)} records for record set '{rsid}'. Columns: {df.columns.tolist()}")

# Choose the primary data table for further analysis (the main protein table)
main_rs_id = None
for rs in metadata.record_sets:
    if 'protein' in rs.name.lower() or 'main' in rs.name.lower():
        main_rs_id = rs.id
        break
if not main_rs_id and len(record_set_ids) > 0:
    main_rs_id = record_set_ids[0]
print(f"\nAnalyzing records in record set: {main_rs_id}")
print(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group on specific numeric fields, illustrating outlier removal and value transformation. All fields are referenced by their `@id` only. Change the `numeric_field_id` and `group_field_id` as appropriate for your dataset.


In [ ]:
# Choose a numeric field and a grouping field by their @id
# For demonstration, try to infer from field ids if not known.
main_df = dataframes[main_rs_id]

# Find likely numeric fields (float/int)
import re
numeric_field_id = None
for f in metadata.record_sets[0].fields:
    # Often molecular_weight, coverage, or peptide_count type fields are numeric
    data_type = getattr(f, 'data_type', None)
    if data_type in ['schema:Number', 'schema:Float', 'schema:Integer'] or (
        re.match(r'.*weight|coverage|score|count|abundance|ratio|peptide|mw|pi', f.name.lower())
    ):
        # Try to use a column with plenty of non-NA and numeric data
        col_id = f.id
        col = main_df.get(col_id)
        if col is not None:
            try:
                main_df[col_id] = pd.to_numeric(col, errors='coerce')
                if main_df[col_id].count() > 0:
                    numeric_field_id = col_id
                    break
            except Exception:
                pass

if not numeric_field_id:
    numeric_field_id = main_df.select_dtypes(float).columns[0] if len(main_df.select_dtypes(float).columns) > 0 else main_df.columns[0]

# Find a grouping field (e.g. a categorical sample/condition or protein id)
group_field_id = None
for f in metadata.record_sets[0].fields:
    if 'sample' in f.name.lower() or 'condition' in f.name.lower() or 'group' in f.name.lower() or 'type' in f.name.lower():
        group_field_id = f.id
        break
if not group_field_id:
    group_field_id = main_df.columns[0]  # Fallback: first column

print(f"Numeric field selected: {numeric_field_id}\nGroup field selected: {group_field_id}")

# Remove records with nulls in analyzable field
analyzable = main_df[numeric_field_id].dropna()
threshold = analyzable.mean()  # Example: above-average
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalization (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group and aggregate (mean, std per group)
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'std', 'count'])
    print(f"\nGrouped statistics by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field and relationships between chosen columns. All visualizations will use columns referenced by their `@id` only.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f'Distribution of {numeric_field_id}')
plt.show()

# If grouping field is categorical with not too many unique values, plot boxplot/group mean
if main_df[group_field_id].nunique() < 25:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and explored the Mass Spectrometry Analysis of Extracellular Vesicles dataset using the `mlcroissant` library.
- Enumerated record sets and their field `@id`s, ensuring all entity references use their unique `@id` identifiers.
- Loaded records into dataframes, filtered and normalized a numeric field, grouped data, and visualized key features of the dataset.

**Next Steps:**
- Try extracting and analyzing other record sets, fields, or perform domain-specific analysis.
- Refer to the [FAIR² documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) for field descriptions.